In [ ]:
import os, sys
import torch, json
import numpy as np

os.chdir("/home/sonat/OCR_HDV/") # TODO: Update to your project path.

from util.slconfig import SLConfig

from datasets import build_dataset
from util.visualizer import COCOVisualizer
from util import box_ops

In [ ]:

dir_path = "logs/"
model_config_path = "config/latin_20class.py"
model_checkpoint_path = "logs/latin_finetune.pth"
nb_class = 20


In [ ]:
if nb_class == 19:
    classes_name = ['word','long','a','b','c','d','e','f','g','h','k','m','n','o','p','q','x','L','others']
elif nb_class == 20:
    classes_name = ['word','long','a','b','c','d','e','f','g','h','k','m','n','o','p','q','x','L','others','symbol']

In [ ]:
from pretraining import build_model_main
args = SLConfig.fromfile(model_config_path) 
args.device = 'cuda' 
device = torch.device(args.device)
model, criterion, postprocessors = build_model_main(args)
checkpoint = torch.load(model_checkpoint_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model.eval()
model.to(device)
print(checkpoint["epoch"])

In [ ]:
args.dataset_file = 'latin_eida'
args.coco_path = "/comp_robot/cv_public_dataset/COCO2017/" # the path of coco
args.fix_size = True 

image_set = "test" # TODO: Update to your desired image set (e.g., "train", "val", "test").
dataset_val = build_dataset(image_set=image_set, args=args)   

In [ ]:
import numpy as np
from shapely.geometry import Polygon
from typing import List, Tuple, Dict, Any, Union
import matplotlib.pyplot as plt
import torch

def compute_iou(pred_poly, gt_poly):
    """Simple IoU computation between two polygons"""
    # Convert to numpy and reshape to (N,2) format

    if torch.is_tensor(pred_poly):
        pred_poly = pred_poly.detach().cpu().numpy()
    if torch.is_tensor(gt_poly):
        gt_poly = gt_poly.detach().cpu().numpy()
    
    # Reshape to (N,2) format
    pred_poly = pred_poly.reshape(-1, 2)
    gt_poly = gt_poly.reshape(-1, 2)
        # Create Shapely polygons
    pred_shapely = Polygon(pred_poly)
    gt_shapely = Polygon(gt_poly)

    # Handle invalid polygons
    if not pred_shapely.is_valid:
        pred_shapely = pred_shapely.buffer(0)
    if not gt_shapely.is_valid:
        gt_shapely = gt_shapely.buffer(0)
        

    intersection = pred_shapely.intersection(gt_shapely).area
    union = pred_shapely.union(gt_shapely).area
    iou = intersection / union if union > 0 else 0

        
    return iou

def match_predictions_independent_detections(
    pred_polygons: List[Any],
    pred_scores: np.ndarray, 
    gt_polygons: List[Any],
    gt_labels: List[int],
    iou_threshold: float = 0.5
) -> Dict[int, List[Dict]]:
    """
    Performs class-aware greedy matching. 
    Returns a dict mapping class_id -> list of detection results.
    """

    if torch.is_tensor(pred_scores): pred_scores = pred_scores.detach().cpu().numpy()
    if torch.is_tensor(pred_polygons): pred_polygons = pred_polygons.detach().cpu().numpy()
    
    # Ensure scores are (N, num_classes)
    num_classes = pred_scores.shape[-1]
    pred_scores = pred_scores.reshape(-1, num_classes)
    
    # Ensure polygons are (N, points_flat) or (N, points, 2)
    num_preds = pred_scores.shape[0]
    pred_polygons = pred_polygons.reshape(num_preds, -1)
    
    num_gts = len(gt_polygons)
    gt_labels_np = np.array(gt_labels, dtype=np.int32)

    # 1. Compute IoU Matrix once for the whole image
    iou_matrix = np.zeros((num_preds, num_gts), dtype=np.float32)
    for i in range(num_preds):
        for j in range(num_gts):
            iou_matrix[i, j] = compute_iou(pred_polygons[i], gt_polygons[j])

    class_results = {c: [] for c in range(num_classes)}
    
    for class_id in range(num_classes):
        gt_indices_C = np.where(gt_labels_np == class_id)[0]
        gt_used_C = np.zeros(len(gt_indices_C), dtype=bool) 
        
        # Collect detections for this class
        detections_C = []
        for i in range(num_preds):
            score = pred_scores[i, class_id]
            if score > 1e-6:
                detections_C.append({"idx": i, "score": score})

        # Sort by score descending
        detections_C.sort(key=lambda x: x["score"], reverse=True)

        for det in detections_C:
            pred_idx = det["idx"]
            score = det["score"]
            
            matched_gt_idx = -1
            max_iou = -1

            if len(gt_indices_C) > 0:
                # Get IoUs for all GTs of this class
                iou_values = iou_matrix[pred_idx, gt_indices_C]
                
                # Sort GT indices by IoU descending
                sorted_gt_local_indices = np.argsort(iou_values)[::-1]
                
                for local_idx in sorted_gt_local_indices:
                    iou = iou_values[local_idx]
                    
                    # If the best remaining IoU is below threshold, stop searching
                    if iou < iou_threshold:
                        break
                    
                    # If this GT is not yet used, claim it!
                    if not gt_used_C[local_idx]:
                        matched_gt_idx = local_idx
                        max_iou = iou
                        break

            if matched_gt_idx != -1:
                gt_used_C[matched_gt_idx] = True
                class_results[class_id].append({
                    "score": score, 
                    "is_tp": 1, 
                    "pred_idx": pred_idx, 
                    "gt_idx": int(gt_indices_C[matched_gt_idx])
                })
            else:
                # No available GT with IoU > threshold
                class_results[class_id].append({
                    "score": score, 
                    "is_tp": 0, 
                    "pred_idx": pred_idx, 
                    "gt_idx": -1
                })

    return class_results

def calculate_interpolated_ap(recall: np.ndarray, precision: np.ndarray) -> float:
    """Calculates AP using the all-point interpolation method."""
    
    # Sort and smooth the precision curve (make it non-increasing)
    for i in range(len(precision) - 2, -1, -1):
        precision[i] = np.maximum(precision[i], precision[i + 1])
        
    # Add start point (0, 1) to the curve for proper integration
    r_to_trapz = np.concatenate(([0.0], recall))
    p_to_trapz = np.concatenate(([1.0], precision))
    
    # Area Under the Curve (Trapezoidal Rule)
    ap = np.trapz(p_to_trapz, r_to_trapz)
    
    return ap

def calculate_mAP(gt_instances_list, pred_instances_list, num_classes: int, iou_threshold: float = 0.5, image_set="val"):
    total_gts_per_class = {i: 0 for i in range(num_classes)}
    class_eval_data = {i: [] for i in range(num_classes)}
    image_matches = []

    for gt, pred in zip(gt_instances_list, pred_instances_list):
        for label in gt['labels']:
            total_gts_per_class[label.item()] += 1
            
        match_info = match_predictions_independent_detections(
            pred['polygons'], pred['scores'], gt['polygons'], gt['labels'], iou_threshold
        )
        image_matches.append(match_info)
        
        for c in range(num_classes):
            for res in match_info[c]:
                # We need these for global F1 and per-class AP
                class_eval_data[c].append(res)

    # --- 1. Global F1 & Threshold Discovery ---
    all_scores = []
    all_is_tp = []
    for c in range(num_classes):
        for res in class_eval_data[c]:
            all_scores.append(res["score"])
            all_is_tp.append(res["is_tp"])
    
    all_scores = np.array(all_scores)
    all_is_tp = np.array(all_is_tp)
    total_gt_global = sum(total_gts_per_class.values())

    # Sort globally by score
    indices = np.argsort(-all_scores)
    tp_cumsum = np.cumsum(all_is_tp[indices])
    fp_cumsum = np.cumsum(1 - all_is_tp[indices])
    
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-10)
    recalls = tp_cumsum / (total_gt_global + 1e-10)
    
    f1_curve = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    best_f1_idx = np.argmax(f1_curve)
    
    # Logic from your notebook: use fixed threshold for test, dynamic for others
    best_threshold = all_scores[indices][best_f1_idx]
    eval_threshold = 0.3969 if image_set == "test" else best_threshold

    # --- 2. Per-Class AP and F1 ---
    ap_per_class = {}
    f1_per_class = {}
    
    for c in range(num_classes):
        results = class_eval_data[c]
        total_gts = total_gts_per_class[c]
        
        if total_gts == 0:
            ap_per_class[c] = 0.0
            f1_per_class[c] = 0.0
            continue
        
        # AP Calculation (Standard Interpolated AP)
        results.sort(key=lambda x: x["score"], reverse=True)
        c_scores = np.array([r["score"] for r in results])
        c_tp = np.array([r["is_tp"] for r in results])
        
        c_tp_sum = np.cumsum(c_tp)
        c_fp_sum = np.cumsum(1 - c_tp)
        c_prec = c_tp_sum / (c_tp_sum + c_fp_sum + 1e-10)
        c_rec = c_tp_sum / total_gts
        ap_per_class[c] = calculate_interpolated_ap(c_rec, c_prec)
        
        # F1 at the specific eval_threshold
        # Count TPs and FPs only for detections above threshold
        top_dets = [r for r in results if r["score"] >= eval_threshold]
        tp_count = sum([r["is_tp"] for r in top_dets])
        fp_count = len(top_dets) - tp_count
        
        p = tp_count / (tp_count + fp_count + 1e-10)
        r = tp_count / (total_gts + 1e-10)
        f1_per_class[c] = 2 * (p * r) / (p + r + 1e-10)

    return {
        "mAP": np.mean(list(ap_per_class.values())),
        "mF1": np.mean(list(f1_per_class.values())),
        "best_threshold": best_threshold,
        "AP_per_class": ap_per_class,
        "F1_per_class": f1_per_class
    }, image_matches

import matplotlib.pyplot as plt
import matplotlib.patches as patches

def display_image_with_boxes(image, polygons, label_ids, gt_polygons, matches, gt_labels, classes_name, save_path="visualization.png"):
    # Create figure and axes
    fig, ax = plt.subplots(1, figsize=(12,12))
    
    # Display the image
    ax.imshow(image)
    matched_preds = [m["pred_idx"] for m in matches if m["gt_idx"] != -1]
    mathed_gts = [m["gt_idx"] for m in matches if m["gt_idx"] != -1]

    for p, polygon in enumerate(polygons):
        if p not in matched_preds:
            box_color = "red"
        else:
            box_color = "darkgreen"
        # Convert list of polygon points to numpy array for easier manipulation
        points = np.array(polygon).reshape(-1, 2)
        # Create a polygon patch
        poly_patch = patches.Polygon(points, linewidth=2., edgecolor=box_color, facecolor='none')
        # Add the patch to the Axes
        ax.add_patch(poly_patch)
        
        if box_color == "darkgreen":
            color = "black"
        else:
            color = "red"
        # Write the class label
        label_id = label_ids[p].item()
        label_str = classes_name[label_id]
        ax.text(
            points[31, 0]-10, points[31, 1] - 10,
            f"{label_str}",
            color=color, fontsize=18,
            bbox=dict(facecolor='white', alpha=0.7, pad=2, edgecolor='none')
        )

    for p, polygon in enumerate(gt_polygons):
        if p not in mathed_gts:
            box_color = "blue"
        else:
            continue
        # Convert list of polygon points to numpy array for easier manipulation
        points = np.array(polygon).reshape(-1, 2)
        # Create a polygon patch
        poly_patch = patches.Polygon(points, linewidth=2., edgecolor=box_color, facecolor='none')
        # Add the patch to the Axes
        ax.add_patch(poly_patch)

        label_id = gt_labels[p].item()
        label_str = classes_name[label_id]
        ax.text(
            points[31, 0]-10, points[31, 1] - 10,
            f"{label_str}",
            color="blue", fontsize=18,
            bbox=dict(facecolor='white', alpha=0.7, pad=2, edgecolor='none')
        )

    ax.axis('off')
    plt.tight_layout()
    # plt.show()
    plt.savefig(save_path, dpi=300)
    plt.close(fig)
    


In [ ]:
def renorm(img: torch.FloatTensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) -> torch.FloatTensor:
    # img: tensor(3,H,W) or tensor(B,3,H,W)
    device = img.device
    dim = img.dim()
    
    assert dim in [3, 4], f"img.dim() should be 3 or 4 but {dim}"
    
    # Ensure mean and std are tensors on the correct device
    mean = torch.tensor(mean, device=device)
    std = torch.tensor(std, device=device)
    
    if dim == 3:
        # Reshape from (3) to (3, 1, 1) to broadcast across H and W
        view_shape = (-1, 1, 1)
        assert img.size(0) == 3, f"Expected 3 channels at dim 0, got {img.size(0)}"
    else:
        # Reshape from (3) to (1, 3, 1, 1) to broadcast across B, H, and W
        view_shape = (1, -1, 1, 1)
        assert img.size(1) == 3, f"Expected 3 channels at dim 1, got {img.size(1)}"

    return img * std.view(view_shape) + mean.view(view_shape)

K=16

gt_instances_list = []
pred_instances_list = []

select_threshold=0.0 # TODO: Update to your desired threshold, especially for visualization (e.g., 0.05 or 0.1).

for j_ in range(len(dataset_val)):
    image, targets = dataset_val[j_]
    with torch.no_grad():
        output = model.cuda()(image[None].cuda())
        polygones = output['pred_boxes']
        label_ids = output['pred_logits'].sigmoid().max(-1)[1]
        
    scores = output['pred_logits'].sigmoid()
    max_scores, _ = scores.max(-1)
    
    # APPLY FILTER HERE (use a low threshold like 0.05 or 0.1)
    keep = max_scores > select_threshold
    
    h, w = image.shape[1:]
    # Only save the boxes that pass the threshold
    filtered_poly = polygones[keep].cpu().detach() * torch.tensor([w, h] * K * 2)
    filtered_scores = scores[keep].cpu().detach()

    gt_polygons = targets['boxes'] * torch.tensor([w,h] * K*2)

    gt_instances_list.append({
        'polygons': gt_polygons.cpu().detach(), 
        'labels': targets['labels'].cpu().detach()
    })
    pred_instances_list.append({
        'polygons': filtered_poly, 
        'scores': filtered_scores
    })

In [ ]:
iou_threshold = 0.5

mAP_results, matches = calculate_mAP(gt_instances_list, pred_instances_list, num_classes=len(classes_name), iou_threshold=iou_threshold, image_set=image_set)
print(f"Final mAP@{iou_threshold}: {mAP_results['mAP']:.4f}, mF1: {mAP_results['mF1']:.4f}, Threshold: {mAP_results['best_threshold']:.4f}")
